In [1]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

In [4]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.drop(columns=['id','Unnamed: 32'],inplace=True)

In [8]:
X_train,X_val,y_train,y_val = train_test_split(df.iloc[:,1:],df.iloc[:,0],test_size=0.2)

In [9]:
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_val = le.transform(y_val)

ss = StandardScaler()
X_train = ss.fit_transform(X_train)
X_val = ss.transform(X_val)

In [10]:
X_train_tensors = torch.from_numpy(X_train.astype(np.float32))
X_val_tensors = torch.from_numpy(X_val.astype(np.float32))
y_train_tensors = torch.from_numpy(y_train.astype(np.float32))
y_val_tensors = torch.from_numpy(y_val.astype(np.float32))

# Dataset


In [12]:
from torch.utils.data import Dataset,DataLoader

class CustomDataset(Dataset):
  def __init__(self,X,y):
    self.features = X
    self.labels = y

  def __len__(self):
    return len(self.features)

  def __getitem__(self, index):
    return self.features[index],self.labels[index]


In [27]:
train_dataset = CustomDataset(X_train_tensors,y_train_tensors)
val_dataset = CustomDataset(X_val_tensors,y_val_tensors)

train_dataloader = DataLoader(train_dataset,shuffle=True,batch_size=10,num_workers=2)
val_dataloader = DataLoader(val_dataset,shuffle=True,batch_size=10,num_workers=2)

# Model Building

In [28]:
import torch.nn as nn

class SimpleNN(nn.Module):

  def __init__(self, input_features):
    super().__init__()
    self.linear1 = nn.Linear(input_features,10)
    self.linear2 = nn.Linear(10,1)
    self.relu = nn.ReLU()
    self.sigmoid = nn.Sigmoid()

  def forward(self,features):
    x = self.linear1(features)
    x = self.relu(x)
    x = self.linear2(x)
    x = self.sigmoid(x)

    return x

In [61]:
model = SimpleNN(X_train.shape[1])

lr = 0.1
epochs = 20

loss_function = nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(),lr=lr)

for _ in range(epochs):
  total_loss=0
  for batch_features,batch_labels in train_dataloader:
    y_pred = model.forward(batch_features).squeeze(1)
    loss = loss_function(y_pred,batch_labels)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    total_loss += loss

  print(f"epoch: {_+1}, loss: {total_loss/len(train_dataloader)}")

epoch: 1, loss: 0.3933908939361572
epoch: 2, loss: 0.14377781748771667
epoch: 3, loss: 0.10270152240991592
epoch: 4, loss: 0.08507278561592102
epoch: 5, loss: 0.0764230415225029
epoch: 6, loss: 0.0723172277212143
epoch: 7, loss: 0.06745200604200363
epoch: 8, loss: 0.06734848767518997
epoch: 9, loss: 0.06325843185186386
epoch: 10, loss: 0.06107833981513977
epoch: 11, loss: 0.06144668161869049
epoch: 12, loss: 0.057206038385629654
epoch: 13, loss: 0.05623187869787216
epoch: 14, loss: 0.05468648299574852
epoch: 15, loss: 0.053728532046079636
epoch: 16, loss: 0.053578879684209824
epoch: 17, loss: 0.05157480016350746
epoch: 18, loss: 0.05007585883140564
epoch: 19, loss: 0.04956858977675438
epoch: 20, loss: 0.048287130892276764


In [59]:
model.eval()  # Set the model to evaluation mode

SimpleNN(
  (linear1): Linear(in_features=30, out_features=10, bias=True)
  (linear2): Linear(in_features=10, out_features=1, bias=True)
  (relu): ReLU()
  (sigmoid): Sigmoid()
)

In [63]:
accuracy_list = []
with torch.no_grad():
  for batch_features,batch_labels in val_dataloader:
    y_pred = model.forward(batch_features)
    y_pred = (y_pred > 0.8).float()

    batch_accuracy = (y_pred.view(-1) == batch_labels).float().mean().item()
    accuracy_list.append(batch_accuracy)

overall_accuracy = sum(accuracy_list) / len(accuracy_list)
print(f'Accuracy: {overall_accuracy:.4f}')

Accuracy: 0.9625
